In [1]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from operator import add
from random import random
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-introduction-preparation").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 14:51:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/11 14:51:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.2


# Spark Introduction

## Why Spark?

Real-world data problems often exceed what a single machine can handle. Spark distributes computation across many cluster nodes -- the same code runs on your laptop and on hundreds of machines in the cloud.

**Core concepts:**
- **Cluster**: A driver node orchestrates work; executor (worker) nodes hold data and compute
- **Partition**: Each worker processes one slice of the data independently
- **Lazy evaluation**: Transformations (`map`, `filter`, …) build a *plan*; actions (`count`, `collect`, `saveAsTextFile`) *execute* it
- **Fault tolerance**: Lost partitions are recomputed from *lineage* -- the recorded chain of transformations

In this notebook we compare pure Python against Spark for two classic problems:  
**Pi estimation** (embarrassingly parallel) and **Word Count** (the MapReduce canonical example).

References:
- https://spark.apache.org/docs/latest/rdd-programming-guide.html
- https://spark.apache.org/examples.html

# Example 1: Computing Pi -- Pure Python

The **Monte Carlo method** estimates π by random sampling:  
- Sample a random point (x, y) in the unit square  
- It lies inside the inscribed circle if x² + y² ≤ 1  
- The probability is π/4, so: **π ≈ 4 × (points inside / total points)**

This version runs sequentially on the driver (single machine).

In [2]:
# Pure Python -- runs on one core
points = 1_000_000

def inside_circle(_):
    x = random() * 2 - 1
    y = random() * 2 - 1
    return 1 if x**2 + y**2 <= 1 else 0

inside = sum(map(inside_circle, range(points)))
pi = 4.0 * inside / points
print(f"Pi is roughly {pi:.6f}")

Pi is roughly 3.144324


# Example 2: Computing Pi -- With Spark

The same algorithm, but `sc.parallelize()` distributes the work across multiple partitions.  
Each partition runs `inside_circle` on its slice independently -- no communication during the map phase.  
Only the final `reduce(add)` requires coordination (a single sum across partitions).

**In a cluster**, the 4 partitions here would run on 4 different machines.

In [3]:
partitions = 4
n = 100_000 * partitions

rdd = sc.parallelize(range(n), partitions)
print("Partitions:", rdd.getNumPartitions())

count = rdd.map(inside_circle).reduce(add)
pi = 4.0 * count / n
print(f"Pi is roughly {pi:.6f}")

Partitions: 4
Pi is roughly 3.151760


In [4]:
# Inspect the execution plan (DAG)
print(rdd.map(inside_circle).toDebugString().decode())

(4) PythonRDD[2] at RDD at PythonRDD.scala:58 []
 |  ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:299 []


# Example 3: Word Count -- With Spark

Word Count is the "Hello World" of distributed computing. It demonstrates the **MapReduce** pattern:

1. `sc.textFile(...)` -- read as an RDD of strings (one per line)
2. `flatMap(split)` -- one line → many words
3. `map(lambda w: (w, 1))` -- each word tagged with count 1
4. `reduceByKey(add)` -- sum all 1s per word (**this causes a shuffle**)
5. `saveAsTextFile(...)` -- write results (triggers the whole pipeline)

`toDebugString()` reveals the **DAG**: notice the `ShuffledRDD` boundary -- that is the reduce phase.

In [5]:
text = sc.textFile(os.path.join(os.getcwd(), "data/US-Constitution.txt"))
print(f"Lines: {text.count()}")

Lines: 717


In [6]:
counts = (text
    .flatMap(lambda line: line.split())
    .map(lambda word: (word.lower(), 1))
    .reduceByKey(add))

print(counts.toDebugString().decode())

(2) PythonRDD[10] at RDD at PythonRDD.scala:58 []
 |  MapPartitionsRDD[9] at mapPartitions at PythonRDD.scala:170 []
 |  ShuffledRDD[8] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[7] at reduceByKey at /tmp/ipykernel_3807/3920608397.py:4 []
    |  PythonRDD[6] at reduceByKey at /tmp/ipykernel_3807/3920608397.py:4 []
    |  /app/work/data-engineering-preparation/data/US-Constitution.txt MapPartitionsRDD[4] at textFile at NativeMethodAccessorImpl.java:0 []
    |  /app/work/data-engineering-preparation/data/US-Constitution.txt HadoopRDD[3] at textFile at NativeMethodAccessorImpl.java:0 []


In [7]:
# Top 10 most frequent words
output = counts.collect()
top10 = sorted(output, key=lambda x: -x[1])[:10]
for word, cnt in top10:
    print(f"{word:20s} {cnt}")

the                  425
of                   297
and                  190
shall                186
be                   128
to                   116
in                   93
or                   79
united               55
a                    53


In [8]:
out = os.path.join(os.getcwd(), "tmp/word_count_result")
counts.saveAsTextFile(out)
print(f"Saved to {out}")

Saved to /app/work/data-engineering-preparation/tmp/word_count_result


# Shutdown Spark when done

In [9]:
spark.stop()